# Precipitation–temperature quantile regression

An alternative to Excel Solver, using the same 102 observations and weighted absolute-error objective.

**Task** · Compare slopes and sensitivities at p=0.5, 0.9 and 0.99.

Run from top to bottom. Predict the result before changing code, then compare it with the output.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

url = 'https://raw.githubusercontent.com/steelpl/lecture/main/CE776-climate-change-carbon-water-cycle/week13/data/precipitation_temperature.csv'
data = pd.read_csv(url)
assert len(data)==102 and (data.precipitation_mm>0).all()
x = data.temperature_c.to_numpy()
y = np.log10(data.precipitation_mm.to_numpy())
X = np.column_stack([np.ones(len(x)), x])
data.head()

Residual r = observed − predicted. Loss is p·r for r≥0 and (1−p)·(−r) for r<0. Writing the residual as u−v, with u,v≥0, gives a linear-programming formulation.

In [ ]:
def quantile_fit(p):
    n = len(y)
    # y - Xb = u - v, u>=0, v>=0; intercept and slope are unrestricted in sign
    cost = np.r_[0., 0., np.full(n,p), np.full(n,1-p)]
    eq = np.column_stack([X, np.eye(n), -np.eye(n)])
    result = linprog(cost, A_eq=eq, b_eq=y,
                     bounds=[(None,None)]*2+[(0,None)]*(2*n), method="highs")
    if not result.success:
        raise RuntimeError(result.message)
    intercept, slope = result.x[:2]
    return {"p":p,"intercept":intercept,"slope":slope,
            "scaling_percent_per_K":100*(10**slope-1),"loss":result.fun}

fits = pd.DataFrame([quantile_fit(p) for p in [.5,.9,.99]])
fits

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7,4))
ax.scatter(x,y,s=16,alpha=.6)
xx = np.linspace(x.min(),x.max(),100)
for row in fits.itertuples():
    ax.plot(xx,row.intercept+row.slope*xx,label=f"q={row.p}")
ax.set(xlabel="Temperature (°C)",ylabel="log10 precipitation (mm)")
ax.legend(); plt.show()

Check whether the p=.99 slope exceeds the p=.9 slope. Explain why very high-quantile sensitivity estimates are uncertain with only 102 observations.

**Reference values** · Approximately 4.8917, 8.7056 and 6.7933%/K. Distinguish sample associations from long-term climate-change effects. Station and period representativeness require separate assessment.